# Tahap 1: Membangun Case Base
**Mata Kuliah:** Penalaran Komputer  
**Domain:** Pidana Umum - Pemalsuan (Pasal 263 & 266 KUHP)  
**Sumber Data:** Direktori Putusan Mahkamah Agung RI (PN Jakarta Utara)

Notebook ini mencakup:
1. Setup struktur folder
2. Konversi PDF → plain text
3. Pembersihan teks
4. Validasi keutuhan teks
5. Logging

## 1. Konfigurasi Path & Setup Folder

In [3]:
import os

BASE_DIR = r'C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan'

PDF_DIR  = os.path.join(BASE_DIR, 'data', 'pdf')
RAW_DIR  = os.path.join(BASE_DIR, 'data', 'raw')
LOG_DIR  = os.path.join(BASE_DIR, 'logs')

for d in [PDF_DIR, RAW_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('Struktur folder siap:')
for d in [PDF_DIR, RAW_DIR, LOG_DIR]:
    print(' ', d)

Struktur folder siap:
  C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\data\pdf
  C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\data\raw
  C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\logs


## Install Library

In [4]:
%pip install pymupdf -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



## 2. Konversi PDF → Plain Text

In [5]:
import fitz
import re

_WATERMARK_SUBSTRINGS_PAGE = [
    'ahkamah agung',
    'blik indonesi',
    'mah agung repub',
    'putusan.mahkamahagung',
]

_WATERMARK_EXACT_LINES = {
    'hkama', 'blik', 'repub',
    'blik indonesi', 'blik indonesia',
    'hkamah', 'hkamah agung', 'hkamah agung repub',
}

_WATERMARK_MAX_LEN = 60

_PAGE_NOISE_PATTERN = re.compile(
    r"""
    direktori\s+putusan\s+mahkamah\s+agung
    | putusan\.mahkamahagung\.go\.id
    | hal\.?\s*\d+\s+dari\s+\d+
    | halaman\s+\d+\s+dari\s+\d+
    | kepaniteraan\s+mahkamah\s+agung
    | email\s*:\s*kepaniteraan@
    | telp\s*:\s*021
    """,
    re.VERBOSE | re.IGNORECASE,
)

MIN_PAGE_CHARS = 30

def _is_noise_line(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False
    if _PAGE_NOISE_PATTERN.search(stripped):
        return True
    if len(stripped) <= _WATERMARK_MAX_LEN:
        if any(sub in stripped.lower() for sub in _WATERMARK_SUBSTRINGS_PAGE):
            return True
        if stripped.lower() in _WATERMARK_EXACT_LINES:
            return True
    return False


def pdf_to_text(pdf_path: str) -> str:
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f'  [ERROR] Gagal membuka {pdf_path}: {e}')
        return ''

    pages = []
    for page in doc:
        blocks = page.get_text("blocks", sort=True)
        lines = []
        for b in blocks:
            if b[6] == 0:
                block_text = b[4].strip()
                if block_text:
                    lines.append(block_text)

        raw_page = page.get_text()

        if len(raw_page.strip()) < MIN_PAGE_CHARS:
            continue

        lines = raw_page.split('\n')
        filtered = [ln for ln in lines if not _is_noise_line(ln)]
        page_text = '\n'.join(filtered).strip()

        if page_text:
            pages.append(page_text)

    doc.close()
    return '\n\n'.join(pages) if pages else ''


pdf_files = sorted([f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')])
print(f'Total PDF ditemukan: {len(pdf_files)}')
for f in pdf_files[:5]:
    print(' ', f)

Total PDF ditemukan: 36
  putusan_1014_pid.b_2019_pn_jkt.utr_20260627114755.pdf
  putusan_1028_pid.b_2020_pn_jkt.utr_20260622233530.pdf
  putusan_1050_pid.b_2021_pn_jkt.utr_20260612003949.pdf
  putusan_1052_pid.b_2021_pn_jkt.utr_20260612004718.pdf
  putusan_1099_pid.b_2020_pn_jkt.utr_20260627121743.pdf


## 3. Pembersihan Teks


In [6]:
from typing import List, Optional, Tuple

DEBUG = False

def dprint(*args, **kwargs):
    if DEBUG:
        print("[DEBUG]", *args, **kwargs)

IDENTITY_LABELS = [
    'nama lengkap', 'nama',
    'tempat lahir', 'tempat/tgl. lahir', 'tempat/tgl lahir',
    'umur/tanggal lahir', 'umur / tanggal lahir', 'umur atau tanggal lahir',
    'umur/tgl lahir',
    'umur/ tgl. lahir',
    'umur/ tgl lahir',
    'umur/ tanggal lahir',
    'umur /tanggal lahir',
    'umur',
    'jenis kelamin',
    'kebangsaan', 'kebangsaan/kewarganegaraan', 'kebangsaan / kewarganegaraan',
    'kewarganegaraan',
    'tempat tinggal', 'alamat',
    'agama', 'pekerjaan', 'pendidikan',
]

_SORTED_LABELS = sorted(IDENTITY_LABELS, key=len, reverse=True)

def _build_label_alt(labels: list) -> str:
    parts = []
    for label in sorted(labels, key=len, reverse=True):
        words = label.split()
        word_pat = []
        for w in words:
            w_escaped = re.escape(w)
            w_escaped = w_escaped.replace(r'\/', r'\s*/\s*')
            word_pat.append(w_escaped + r'\.?')
        parts.append(r'\s+'.join(word_pat))
    return '|'.join(parts)

_LABEL_ALT = _build_label_alt(IDENTITY_LABELS)

_NUM_ONLY_LINE = re.compile(r'^\s*([1-9][0-9]?|[ivxIVX]{1,4})(\.?)\s*$')

_LABEL_LINE = re.compile(
    rf'^\s*(?:'
    rf'(?:(?P<num>[1-9][0-9]?)(?P<numdot>\.?)|(?:[ivxIVX]{{1,4}}\.?))'
    rf'\s*(?:\.{{2,}}\s*)?\s+'
    rf')?(?P<label>{_LABEL_ALT})\s*'
    rf'(?P<rest>:.*)?$',
    re.IGNORECASE,
)

_LABEL_LINE_LOOSE = re.compile(
    rf'^\s*(?:'
    rf'(?:(?:[1-9][0-9]?)\.?\s+)|(?:[ivxIVX]{{1,4}}\.?\s+)'
    rf')?(?P<label>{_LABEL_ALT})\b',
    re.IGNORECASE,
)

_INLINE_LABEL_WITH_COLON = re.compile(
                rf'^\s*(?:[1-9][0-9]?\.?\s+)?(?:{_LABEL_ALT})\s*:',
                re.IGNORECASE
            )

_COLON_ONLY_LINE = re.compile(r'^[\s\u00a0]*:[\s\u00a0]*$')

_TRAILING_NUM = re.compile(r'^(?P<value>.*\S)\s+(?P<num>[1-9][0-9]?)(?P<numdot>\.?)\s*$')

_END_MARKERS = re.compile(
    r'(?:'
    r'\bterdakwa\b[\s\S]{0,100}?\b(ditangkap|ditahan|tidak\s+ditahan|menghadap)\b'
    r'|'
    r'\btelah\s+ditahan\b'
    r'|'
    r'\bditahan\s+berdasarkan\b'
    r')',
    re.IGNORECASE,
)

_REQUIRED_BLOCK_LABELS = (
    'nama lengkap', 'nama',
    'tempat lahir', 'tempat/tgl. lahir', 'tempat/tgl lahir',
    'umur/tanggal lahir', 'umur / tanggal lahir', 'umur atau tanggal lahir',
    'umur/tgl lahir',
    'jenis kelamin',
)

_IDENTITY_SEARCH_WINDOW = 6000

_NAME_LABEL_ALIASES = {'nama', 'nama lengkap'}

_TERDAKWA_IDENTIFIER = re.compile(
    r'^\s*(?:terdakwa\s+)?(?:[ivx]+|[0-9]+)\s*[.)]?\s*(?:nama|nama\s+lengkap)\s*[:\.]',
    re.IGNORECASE
)

_TERDAKWA_SECTION_MARKER = re.compile(
    r'^\s*(?:para\s+)?terdakwa\s+(?:[ivx]+|[0-9]+)\s*$',
    re.IGNORECASE,
)

_DETENTION_LIST_MARKER = re.compile(
    r'^([\s\u00a0]*(?:[a-z]|[1-9][0-9]?)\.)[\s\u00a0]*$',
    re.MULTILINE,
)

_DEFENDANT_SECTION_MARKER = re.compile(
    r'^\s*(?:'
    r'(?:para\s+)?terdakwa\s+(?:[ivx]+|[0-9]+)'
    r'|'
    r'(?:[ivx]+|[0-9]+)\.\s*(?:nama|nama\s+lengkap)\s*:'
    r')\s*$',
    re.IGNORECASE,
)

_LABEL_WITH_TRAILING_VALUE = re.compile(
    rf'^\s*(?P<num>[1-9][0-9]?)\.?\s+(?P<label>{_LABEL_ALT})\s+(?P<value>\S.*)',
    re.IGNORECASE,
)


def _canonical_label(label: str) -> str:
    label = re.sub(r'\s+', ' ', label.strip().lower())
    if label in _NAME_LABEL_ALIASES:
        return 'nama'
    return label


def _is_label_line(line: str) -> bool:
    result = bool(_LABEL_LINE.match(line))
    return result


def _is_end_marker(line: str) -> bool:
    result = bool(_END_MARKERS.search(line))
    return result


def _format_num(num: Optional[str], numdot: str) -> str:
    if not num:
        return ''
    return f"{num}{numdot} "


def _normalize_label_text(line: str) -> Optional[str]:
    m = _LABEL_LINE_LOOSE.match(line)
    if m:
        return _canonical_label(m.group('label'))
    return None


def _fix_label_trailing_value(text: str) -> str:
    lines = text.split('\n')
    out = []

    LABEL_CONTINUATION_WORDS = {'lengkap', 'lahir', 'kelamin', 'islam', 'negara'}

    for ln in lines:
        m = _LABEL_WITH_TRAILING_VALUE.match(ln)
        if m:
            val = m.group('value').strip()
            lbl = m.group('label').strip().lower()

            combined = f"{lbl} {val.lower()}"
            if combined in [l.lower() for l in IDENTITY_LABELS]:
                out.append(ln)
                continue

            if val.lower() in LABEL_CONTINUATION_WORDS:
                out.append(ln)
                continue

            if not _LABEL_LINE.match(val) and not val.startswith(':'):
                fixed = f"{m.group('num')}. {m.group('label')} : {val}"
                dprint(f"  _fix_label_trailing_value: {ln!r} → {fixed!r}")
                out.append(fixed)
                continue
        out.append(ln)
    return '\n'.join(out)


def _fix_label_with_embedded_value(text: str) -> str:
    lines = text.split('\n')
    out = []
    for ln in lines:
        m = _LABEL_WITH_TRAILING_VALUE.match(ln)
        if m:
            val = m.group('value').strip()
            lbl = m.group('label').strip().lower()
            combined = f"{lbl} {val.lower()}"

            label_end = m.start('value')
            prefix = ln[:label_end]

            colon_count = prefix.count(':')
            if colon_count >= 1:
                out.append(ln)
                continue

            if re.match(r'^:+\s*$', val):
                out.append(ln)
                continue

            if val.startswith(':'):
                out.append(ln)
                continue

            if lbl in [l.lower() for l in IDENTITY_LABELS] and combined not in [l.lower() for l in IDENTITY_LABELS]:
                fixed = f"{m.group('num')}. {m.group('label')} : {val}"
                dprint(f"  _fix_label_with_embedded_value: {ln!r} → {fixed!r}")
                out.append(fixed)
                continue
        out.append(ln)
    return '\n'.join(out)


def _join_split_labels(text: str) -> str:
    lines = text.split('\n')
    out = []
    i = 0
    while i < len(lines):
        ln = lines[i]
        if ln.rstrip().endswith('/') and _LABEL_LINE_LOOSE.match(ln):
            j = i + 1
            buf = ln.rstrip()
            while j < len(lines):
                nxt = lines[j].strip()
                if not nxt:
                    j += 1
                    continue
                buf = buf + nxt
                j += 1
                if ':' in nxt:
                    break
                if j < len(lines) and _LABEL_LINE.match(lines[j].strip()):
                    break
            dprint(f"  _join_split_labels: merged → {buf!r}")
            out.append(buf)
            i = j
        else:
            out.append(ln)
            i += 1
    return '\n'.join(out)


def join_detention_list_markers(text: str) -> str:
    lines = text.split('\n')
    out: List[str] = []
    i = 0
    n = len(lines)
    while i < n:
        ln = lines[i]
        m = _DETENTION_LIST_MARKER.match(ln)
        if m:
            marker = m.group(1)
            j = i + 1
            collected = []
            while j < n:
                nxt = lines[j]
                if nxt.strip() == '':
                    j += 1
                    continue
                if _DETENTION_LIST_MARKER.match(nxt):
                    break
                collected.append(nxt.strip())
                j += 1
                if len(collected) >= 1 and (
                    nxt.rstrip().endswith(';') or nxt.rstrip().endswith(':')
                ):
                    break
            if collected:
                out.append(f"{marker} {' '.join(collected)}")
                i = j
                continue
        out.append(ln)
        i += 1
    return '\n'.join(out)


def _collect_label_block(lines: List[str], start: int) -> Tuple[List[Tuple], int]:
    labels = []
    i = start
    n = len(lines)

    while i < n:
        ln = lines[i]
        stripped = ln.strip()

        if stripped == '' or _COLON_ONLY_LINE.match(ln):
            i += 1
            continue

        if _is_end_marker(ln):
            break

        num_only = _NUM_ONLY_LINE.match(stripped)
        if num_only:
            j = i + 1
            while j < n and lines[j].strip() == '':
                j += 1
            if j < n:
                candidate = lines[j]
                candidate_stripped = re.sub(r'[.\s]+$', '', candidate)
                next_m = _LABEL_LINE.match(candidate) or _LABEL_LINE.match(candidate_stripped)
                if next_m and (next_m.group('rest') is None or
                              re.sub(r'^:\s*', '', next_m.group('rest') or '').strip() == ''):
                    labels.append((next_m.group('label'), num_only.group(1), num_only.group(2) or ''))
                    dprint(f"  [{i}+{j}] num+label: {next_m.group('label')!r}")
                    i = j + 1
                    continue
                elif j < n and _LABEL_LINE.match(lines[j]):
                    next_m2 = _LABEL_LINE.match(lines[j])
                    rest2 = next_m2.group('rest') or ''
                    val2 = re.sub(r'^:\s*', '', rest2).strip()
                    if val2 == '':
                        labels.append((next_m2.group('label'), num_only.group(1), num_only.group(2) or ''))
                        dprint(f"  [{i}+{j}] num+label(colon): {next_m2.group('label')!r}")
                        i = j + 1
                        continue
            dprint(f"  [{i}] num-only tanpa label berikut, stop")
            break

        m = _LABEL_LINE.match(ln)
        if m:
            rest = m.group('rest') or ''
            value_inline = re.sub(r'^:\s*', '', rest).strip()
            if value_inline == '':
                labels.append((m.group('label'), m.group('num'), m.group('numdot') or ''))
                dprint(f"  [{i}] label: {m.group('label')!r}")
                i += 1
                continue
            else:
                dprint(f"  [{i}] label dengan value inline '{value_inline}' → stop")
                break

        break

    return labels, i


def _collect_values(lines: List[str], start: int, n_needed: int) -> Tuple[List[str], int]:
    i = start
    n = len(lines)
    raw = []

    while i < n:
        ln = lines[i]
        stripped = ln.strip()

        if stripped == '':
            i += 1
            continue
        if _is_end_marker(ln):
            dprint(f"  [{i}] value collect: end marker")
            break
        if _COLON_ONLY_LINE.match(ln):
            dprint(f"  [{i}] value collect: colon-only skip")
            i += 1
            continue
        if _is_label_line(ln):
            dprint(f"  [{i}] value collect: label baru, stop")
            break
        if _TERDAKWA_SECTION_MARKER.match(stripped):
            dprint(f"  [{i}] value collect: terdakwa section marker, stop")
            break
        if _NUM_ONLY_LINE.match(stripped):
            j = i + 1
            while j < n and lines[j].strip() == '':
                j += 1
            if j < n and _is_label_line(lines[j]):
                dprint(f"  [{i}] value collect: num-only sebelum label, stop")
                break

        val = stripped.rstrip(';').strip()

        if re.match(r'^para\s*$', val, re.IGNORECASE) and len(raw) >= 1:
            dprint(f"  [{i}] value collect: 'para' orphan, stop")
            break

        while val.endswith(','):
            j = i + 1
            while j < n and lines[j].strip() == '':
                j += 1
            if j >= n:
                break
            nxt = lines[j].strip()
            if (_is_label_line(lines[j])
                    or _is_end_marker(lines[j])
                    or _TERDAKWA_SECTION_MARKER.match(nxt)
                    or _COLON_ONLY_LINE.match(lines[j])):
                break
            val = val + ' ' + nxt.rstrip(';').strip()
            dprint(f"  [{i}] value trailing-comma continuation [{j}]: {val!r}")
            i = j

        raw.append(val)
        dprint(f"  [{i}] value: {val!r}")
        i += 1

        if len(raw) >= n_needed * 3:
            break

    return raw, i


def _merge_raw_values(raw: List[str], n_labels: int, labels: List[Tuple]) -> Optional[List[str]]:
    if len(raw) == n_labels:
        return raw

    if len(raw) < n_labels:
        return None

    alamat_indices = [
        idx for idx, (lbl, _, _) in enumerate(labels)
        if 'tinggal' in lbl.lower() or 'alamat' in lbl.lower()
    ]

    result = list(raw)
    while len(result) > n_labels:
        if alamat_indices:
            ai = alamat_indices[0]
            if ai + 1 < len(result):
                extra = result.pop(ai + 1)
                result[ai] = result[ai] + ' ' + extra
                dprint(f"  merge ke alamat[{ai}]: {result[ai]!r}")
            else:
                extra = result.pop(-1)
                result[-1] = result[-1] + ' ' + extra
        else:
            extra = result.pop(-1)
            result[-1] = result[-1] + ' ' + extra

    return result if len(result) == n_labels else None


def _is_two_column_block(lines: List[str]) -> bool:
    inline_value_count = 0
    total_labels = 0

    for ln in lines:
        m = _LABEL_LINE.match(ln)
        if m:
            total_labels += 1
            rest = m.group('rest') or ''
            if re.sub(r'^:\s*', '', rest).strip() != '':
                inline_value_count += 1

    if total_labels > 0 and (inline_value_count / total_labels) > 0.5:
        return False

    label_count = 0
    i = 0
    n = len(lines)
    while i < n:
        ln = lines[i]
        stripped = ln.strip()
        if stripped == '' or _COLON_ONLY_LINE.match(ln):
            i += 1
            continue
        if _is_end_marker(ln):
            break
        num_only = _NUM_ONLY_LINE.match(stripped)
        if num_only:
            j = i + 1
            while j < n and lines[j].strip() == '':
                j += 1
            if j < n and _is_label_line(lines[j]):
                nxt = _LABEL_LINE.match(lines[j])
                if nxt and (nxt.group('rest') is None or re.sub(r'^:\s*', '', nxt.group('rest') or '').strip() == ''):
                    label_count += 1
                    i = j + 1
                    continue
            break
        m = _LABEL_LINE.match(ln)
        if m:
            rest = m.group('rest') or ''
            if re.sub(r'^:\s*', '', rest).strip() == '':
                label_count += 1
                i += 1
                continue
            else:
                break
        break
    return label_count >= 2


def _try_parse_semicolon_value_block(lines: List[str]) -> Optional[List[tuple]]:
    labels = []
    i = 0
    n = len(lines)
    while i < n:
        stripped = lines[i].strip()
        if stripped == '':
            i += 1
            continue
        m = _LABEL_LINE.match(lines[i])
        if m:
            rest = m.group('rest') or ''
            val = re.sub(r'^:\s*', '', rest).strip()
            if val == '' or re.match(r'^:+\s*$', val):
                labels.append(m.group('label'))
                i += 1
                continue
        break

    if len(labels) < 3:
        return None
    while i < n and re.match(r'^[\s:]*$', lines[i]):
        i += 1
    if i >= n:
        return None

    remainder = ' '.join(ln.strip() for ln in lines[i:] if ln.strip())
    remainder = re.sub(r'^[:\s]+', '', remainder)
    raw_values = [v.strip() for v in remainder.split(';') if v.strip()]

    if len(raw_values) < len(labels):
        return None

    tinggal_idx = next(
        (idx for idx, l in enumerate(labels) if 'tinggal' in l.lower() or 'alamat' in l.lower()),
        None
    )

    values = list(raw_values)
    while len(values) > len(labels):
        if tinggal_idx is not None and tinggal_idx + 1 < len(values):
            extra = values.pop(tinggal_idx + 1)
            values[tinggal_idx] = values[tinggal_idx] + '; ' + extra
        else:
            extra = values.pop(-1)
            values[-1] = values[-1] + '; ' + extra

    return list(zip(labels, values[:len(labels)]))


def _try_parse_two_column_block(lines: List[str]) -> Optional[List[tuple]]:
    dprint(f"_try_parse_two_column_block: {len(lines)} baris")

    if not _is_two_column_block(lines):
        dprint("  bukan two-column (quick check)")
        result = _try_parse_semicolon_value_block(lines)
        if result is not None:
            dprint("  semicolon-value block terdeteksi")
            return result
        return None

    labels, value_start = _collect_label_block(lines, 0)
    dprint(f"  {len(labels)} label collected, value start={value_start}")

    if len(labels) < 2:
        dprint("  terlalu sedikit label")
        return None

    raw_values, _ = _collect_values(lines, value_start, len(labels))
    dprint(f"  {len(raw_values)} raw values")

    merged = _merge_raw_values(raw_values, len(labels), labels)
    if merged is None:
        dprint(f"  MISMATCH: {len(raw_values)} raw vs {len(labels)} labels")
        return None

    result = []
    for (label, num, numdot), value in zip(labels, merged):
        result.append((label, value))
        dprint(f"  PAIR: {label!r} -> {value!r}")

    return result


def _strip_label_trailing_dot(text: str) -> str:
    lines = text.split('\n')
    out = []
    for ln in lines:
        stripped = ln.rstrip()
        if stripped.endswith('.'):
            candidate = stripped[:-1].rstrip()
            if _LABEL_LINE.match(candidate + ' :'):
                out.append(candidate)
                continue
        out.append(ln)
    return '\n'.join(out)


def find_identity_block_span(text: str) -> Optional[Tuple[int, int]]:
    window = text[:_IDENTITY_SEARCH_WINDOW]
    lines = window.split('\n')

    required_set = {l.lower() for l in _REQUIRED_BLOCK_LABELS}

    offsets = []
    pos = 0
    for ln in lines:
        offsets.append(pos)
        pos += len(ln) + 1

    matches = []
    for li, ln in enumerate(lines):
        m = _LABEL_LINE.match(ln)
        if m and re.sub(r'\s+', ' ', m.group('label').lower().strip()) in required_set:
            matches.append((li, offsets[li], m.group('label').lower()))

    dprint(f"find_identity_block_span: {len(matches)} required-label matches ditemukan")
    for li, off, lbl in matches:
        dprint(f"  line {li} offset {off}: {lbl!r}")

    if not matches:
        dprint("  -> return None (tidak ada required label)")
        return None

    first_li = matches[0][0]
    first_match_offset = matches[0][1]

    if first_li > 0 and _NUM_ONLY_LINE.match(lines[first_li - 1]):
        dprint(f"  baris sebelum first match adalah num-only, offset digeser ke {offsets[first_li - 1]}")
        first_match_offset = offsets[first_li - 1]

    last_match_offset = matches[-1][1]
    dprint(f"  span awal: first_offset={first_match_offset}, last_offset={last_match_offset}")

    tail = text[last_match_offset:]
    end_m = _END_MARKERS.search(tail)
    if end_m is None:
        dprint("  -> return None (tidak ada END_MARKER setelah last label)")
        return None

    end_offset = last_match_offset + end_m.start()
    dprint(f"  END_MARKER ditemukan di offset {end_offset}, snippet: {text[end_offset:end_offset+60]!r}")
    return (first_match_offset, end_offset)


def split_merged_label_lines(text: str) -> str:
    lines = text.split('\n')
    out: List[str] = []

    label_with_colon = re.compile(
        rf'(?:^|(?<=[.\s]))(?:(?:\d{{1,2}}|[ivxlIVXL]{{1,4}})\.?\s+)?\b({_LABEL_ALT})\b\s*:',
        re.IGNORECASE
    )

    for line in lines:
        all_matches = list(label_with_colon.finditer(line))

        if not all_matches:
            out.append(line)
            continue

        seen_labels = set()
        first_label = _canonical_label(all_matches[0].group(1).lower())
        seen_labels.add(first_label)

        split_positions = []
        for m in all_matches[1:]:
            lbl = _canonical_label(m.group(1).lower())
            if lbl not in IDENTITY_LABELS:
                continue
            if lbl in seen_labels:
                continue
            split_positions.append(m.start())
            seen_labels.add(lbl)

        if not split_positions:
            out.append(line)
            continue

        dprint(f"split_merged_label_lines: split baris di posisi {split_positions}: {line!r}")
        pieces = []
        prev = 0
        for pos in split_positions:
            pieces.append(line[prev:pos].rstrip())
            prev = pos
        pieces.append(line[prev:])
        out.extend(pieces)

    return '\n'.join(out)


from typing import List, Tuple
import re

def normalize_identity_lines(text: str) -> str:
    dprint("\n=== normalize_identity_lines: START ===")
    text = _strip_label_trailing_dot(text)
    text = _fix_label_trailing_value(text)
    text = _fix_label_with_embedded_value(text)
    text = _join_split_labels(text)
    text = split_concatenated_labels_no_colon(text)
    text = split_merged_label_lines(text)
    lines = text.split('\n')
    n = len(lines)

    dprint(f"  total baris setelah split_merged: {n}")
    for i, ln in enumerate(lines):
        dprint(f"  raw[{i:3d}] {ln!r}")

    def _split_into_defendant_segments(lines: List[str]) -> List[Tuple[str, List[str]]]:
        segments = []
        current_header = ''
        current_lines = []
        for ln in lines:
            if _DEFENDANT_SECTION_MARKER.match(ln.strip()):
                if current_lines or current_header:
                    segments.append((current_header, current_lines))
                current_header = ln
                current_lines = []
            else:
                current_lines.append(ln)
        if current_lines or current_header:
            segments.append((current_header, current_lines))
        return segments

    dprint(f"\n  --- mencoba deteksi two-column block ---")

    dprint(f"  lines sebelum segment split:")
    for i, ln in enumerate(lines):
        dprint(f"    [{i}] {ln!r}")

    segments = _split_into_defendant_segments(lines)
    has_multiple_segments = len(segments) > 1

    if has_multiple_segments:
        dprint(f"  multi-defendant: {len(segments)} segmen ditemukan")
        rebuilt_parts = []
        for seg_header, seg_lines in segments:
            if seg_header:
                rebuilt_parts.append(seg_header)
            if not seg_lines:
                continue
            two_col = _try_parse_two_column_block(seg_lines)
            if two_col is not None:
                dprint(f"  TWO-COLUMN terdeteksi di segmen {seg_header!r}")
                for label, value in two_col:
                    rebuilt_parts.append(f"{label} : {value}")
            else:
                seg_text = '\n'.join(seg_lines)
                normalized_seg = normalize_identity_lines(seg_text)
                rebuilt_parts.append(normalized_seg)
        result = '\n'.join(rebuilt_parts)
        dprint("=== normalize_identity_lines: END (multi-defendant path) ===\n")
        return result

    two_col_result = _try_parse_two_column_block(lines)
    if two_col_result is not None:
        dprint("  TWO-COLUMN terdeteksi, rekonstruksi baris")
        rebuilt = []
        for label, value in two_col_result:
            rebuilt.append(f"{label} : {value}")
            dprint(f"    rebuilt: {label!r} : {value!r}")
        result = '\n'.join(rebuilt)
        dprint("=== normalize_identity_lines: END (two-column path) ===\n")
        return result

    dprint("  bukan two-column, lanjut ke logika normalize biasa")

    paired: List[str] = []
    i = 0
    while i < n:
        line = lines[i]

        if line.strip() == '':
            paired.append(line)
            i += 1
            continue

        m = _LABEL_LINE.match(line)
        if m:
            num, numdot, label, rest = m.group('num'), m.group('numdot') or '', m.group('label'), m.group('rest')
            dprint(f"  paired loop [{i}]: LABEL num={num!r} label={label!r} rest={rest!r}")
            j = i + 1
            if rest is None:
                while j < n and lines[j].strip() == '':
                    j += 1
                next_line_stripped = lines[j].strip() if j < n else ''

                if j < n and re.match(r'^:$', next_line_stripped):
                    j += 1
                    while j < n and lines[j].strip() == '':
                        j += 1
                    if j < n and _is_label_line(lines[j]):
                        rest = ':'
                    else:
                        value_start = lines[j].strip() if j < n else ''
                        rest = ': ' + value_start
                        j += 1
                elif j < n and next_line_stripped.startswith(':'):
                    rest = next_line_stripped
                    j += 1
                elif j < n and _is_label_line(lines[j]):
                    rest = ':'
                else:
                    rest = ':'
            rest = re.sub(r'^:\s+', ': ', rest) if rest != ':' else rest
            current = f"{_format_num(num, numdot)}{label} {rest}".rstrip()
            dprint(f"    -> paired: {current!r}")
            paired.append(current)
            i = j
            continue

        paired.append(line)
        i += 1

    non_blank = [ln for ln in paired if ln.strip() != '']
    p = len(non_blank)

    dprint(f"\n  --- fields pass (non_blank={p}) ---")
    fields: List[str] = []
    idx = 0

    structural_interrupt = re.compile(r'^\s*\d{1,2}\s*[\.\)]\s*\w+.*?:\s*')

    while idx < p:
        line = non_blank[idx]

        num_only = _NUM_ONLY_LINE.match(line)
        if num_only and idx + 1 < p and _is_label_line(non_blank[idx + 1]):
            nxt_m = _LABEL_LINE.match(non_blank[idx + 1])
            if nxt_m and not nxt_m.group('num'):
                num, numdot = num_only.group(1), num_only.group(2)
                idx += 1
                line = f"{num}{numdot} {non_blank[idx].lstrip()}"
                dprint(f"  fields [{idx}]: num-only merge -> {line!r}")
                idx += 1
            else:
                idx += 1
                continue
        elif _is_label_line(line) and not _is_end_marker(line):
            dprint(f"  fields [{idx}]: label line {line!r}")
            idx += 1
        else:
            fields.append(line)
            dprint(f"  fields [{idx}]: non-label, append as-is {line!r}")
            idx += 1
            continue

        buf = line
        buf_value = re.sub(rf'^\s*(?:\d{{1,2}}\.?\s+)?(?:{_LABEL_ALT})\s*:\s*', '', buf, flags=re.IGNORECASE).strip()
        dprint(f"    buf={buf!r}, extracted value={buf_value!r}")
        if not buf_value:
            fields.append(buf)
            dprint(f"    -> fields append (no value): {buf!r}")
            continue

        lm = _LABEL_LINE.match(buf)
        current_label = _canonical_label(lm.group('label')) if lm else ''

        while idx < p:
            nxt = non_blank[idx]

            if _INLINE_LABEL_WITH_COLON.match(nxt) or structural_interrupt.match(nxt):
                dprint(f"    continuation stop [{idx}]: structural interrupt / inline label terdeteksi -> {nxt!r}")
                break

            if nxt.strip().startswith(':'):
                clean_nxt_val = re.sub(r'[^a-zA-Z0-9]', '', nxt).lower()
                clean_buf_val = re.sub(r'[^a-zA-Z0-9]', '', buf_value).lower()

                if clean_nxt_val and (clean_nxt_val in clean_buf_val or clean_buf_val in clean_nxt_val):
                    dprint(f"    continuation stop [{idx}]: duplikasi nilai murni terdeteksi -> {nxt!r}")
                    idx += 1
                    break

            end_check = _is_end_marker(nxt)
            label_check = _is_label_line(nxt)

            if end_check or label_check:
                dprint(f"    continuation stop at [{idx}]: end={end_check} label={label_check}")
                break
            if _TERDAKWA_IDENTIFIER.match(nxt):
                dprint(f"    continuation stop at [{idx}]: terdakwa identifier")
                break
            if _NUM_ONLY_LINE.match(nxt) and idx + 1 < p and _is_label_line(non_blank[idx + 1]):
                dprint(f"    continuation stop at [{idx}]: num-only before label")
                break

            buf = buf.rstrip() + ' ' + nxt.strip()
            dprint(f"    continuation [{idx}]: buf={buf!r}")
            idx += 1

        fields.append(buf)
        dprint(f"    -> fields append final: {buf!r}")

    dprint(f"\n  --- final pass (fields={len(fields)}) ---")
    final: List[str] = []
    f_idx = 0
    fn = len(fields)
    while f_idx < fn:
        current = fields[f_idx]
        if f_idx + 1 < fn:
            tm = _TRAILING_NUM.match(current)
            nxt_m = _LABEL_LINE.match(fields[f_idx + 1]) if tm else None
            if tm and nxt_m and _is_label_line(fields[f_idx + 1]) and not nxt_m.group('num'):
                dprint(f"  final [{f_idx}]: trailing-num split {current!r}")
                final.append(tm.group('value'))
                num, numdot = tm.group('num'), tm.group('numdot')
                final.append(f"{num}{numdot} {fields[f_idx + 1].lstrip()}")
                f_idx += 2
                continue
        final.append(current)
        f_idx += 1

    cleaned_final = []
    for line in final:
        if re.search(r'^\s*\d{1,2}\s*[\.\)]\s*nama\s*:\s*lengkap\s*:', line, re.IGNORECASE):
            line = re.sub(r'nama\s*:\s*lengkap', 'nama lengkap', line, flags=re.IGNORECASE)
            line = re.sub(r':\s*:', ':', line)
        cleaned_final.append(line)

    joined = '\n'.join(final)
    if text.endswith('\n') and not joined.endswith('\n'):
        joined += '\n'

    dprint("=== normalize_identity_lines: END (normal path) ===\n")
    return joined


def split_concatenated_labels_no_colon(text: str) -> str:
    pattern = re.compile(
        rf'(?:^|(?<=\n))(\d{{1,2}}\.\s*(?:{_LABEL_ALT}))\.\s+({_LABEL_ALT})\s*$',
        re.IGNORECASE | re.MULTILINE,
    )
    def repl(m):
        return f"{m.group(1)}\n{m.group(2)}"
    return pattern.sub(repl, text)


def clean_identity_block(text: str) -> str:
    dprint("\n=== clean_identity_block: START ===")
    span = find_identity_block_span(text)

    if span is None:
        dprint("  span=None, return text as-is")
        return text

    start, end = span
    dprint(f"  span=({start}, {end})")
    dprint(f"  head: {text[:start]!r}")
    dprint(f"  block (first 200): {text[start:start+200]!r}")
    dprint(f"  tail (first 100): {text[end:end+100]!r}")

    head, block, tail = text[:start], text[start:end], text[end:]

    normalized = normalize_identity_lines(block)

    if "ditahan dalam tahanan" in tail.lower() and "terdakwa" in tail.lower():
        match = re.search(r'^(.*?ditahan dalam tahanan.*?:)\s*', tail, flags=re.IGNORECASE | re.DOTALL)
        if match:
            first_tail_line = match.group(1).strip()

            if first_tail_line.lower() not in normalized.lower():
                dprint(f"   [CASE_012 RECOVERY] Mengembalikan baris penahanan: {first_tail_line!r}")

                normalized = normalized.strip() + "\n" + first_tail_line
                tail = tail[match.end():]

    dprint(f"  normalized block:\n{normalized}")
    dprint("=== clean_identity_block: END ===\n")
    return head + normalized + tail

In [7]:
import unicodedata

_WATERMARK_SUBSTRINGS = [
    'ahkamah agung',
    'mah agung repub',
    'putusan.mahkamahagung',
]

_WATERMARK_MAX_LINE_LEN = 60

HEADER_FOOTER_PATTERNS = [
    r"putusan\.mahkamahagung\.go\.id[^\n]*",
    r"halaman\s+\d+\s+dari\s+\d+\s*hal\.?",
    r"halaman\s+\d+$",
    r"hal\.?\s*\d+\s+dari\s+\d+\s*hal\.?",
    r"pelaksanaan\s+fungsi\s+peradilan\..*?(?=\n\s*\n|\Z)",
    r"dalam\s+hal\s+anda\s+menemukan\s+inakurasi.*?(?=\n\s*\n|\Z)",
    r"email\s*:\s*kepaniteraan@mahkamahagung[^\n]*",
    r"telp\s*:\s*021[^\n]*",
    r"disclaimer\s*:?[^\n]*",
    r"\bpid\.i\.a\.\d+\b",
    r"hal\.\s*\d+\s+putusan\s+no\.[^\n]*",
]


def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r'[\u200b\u200c\u200d\ufeff\u00ad]', '', text)
    return text


def remove_watermark_lines(text: str) -> str:
    lines = text.split('\n')
    result = []
    for ln in lines:
        stripped = ln.strip()
        if not stripped:
            result.append(ln)
            continue
        if stripped in _WATERMARK_EXACT_LINES:
            continue
        if len(stripped) <= _WATERMARK_MAX_LINE_LEN:
            if any(sub in stripped.lower() for sub in _WATERMARK_SUBSTRINGS):
                continue
        result.append(ln)
    return '\n'.join(result)


def remove_headers_footers(text: str) -> str:
    dotall_patterns = [
        r"putusan\.mahkamahagung\.go\.id[^\n]*",
        r"halaman\s+\d+\s+dari\s+\d+\s*hal\.?",
        r"hal\.?\s*\d+\s+dari\s+\d+\s*hal\.?",
        r"email\s*:\s*kepaniteraan@mahkamahagung[^\n]*",
        r"telp\s*:\s*021[^\n]*",
        r"disclaimer\s*:?[^\n]*",
        r"\bpid\.i\.a\.\d+\b",
    ]

    multiline_only_patterns = [
        r"halaman\s+\d+$",
        r"pelaksanaan\s+fungsi\s+peradilan\..*?(?=\n\s*\n|\Z)",
        r"dalam\s+hal\s+anda\s+menemukan\s+inakurasi.*?(?=\n\s*\n|\Z)",
    ]
    for pattern in dotall_patterns:
        text = re.sub(pattern, '', text, flags=re.MULTILINE | re.IGNORECASE | re.DOTALL)
    for pattern in multiline_only_patterns:
        text = re.sub(pattern, '', text, flags=re.MULTILINE | re.IGNORECASE)
    return text


def fix_broken_label(text: str) -> str:
    broken_labels = [
        (r'kebangsaan/kewarganeg\w*\s*\n\s*\w+', 'kebangsaan/kewarganegaraan'),
        (r'kebangsaan\s*/\s*kewarganeg\w*\s*\n\s*\w+', 'kebangsaan / kewarganegaraan'),
    ]
    for pattern, replacement in broken_labels:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


def fix_double_colon(text: str) -> str:
    return re.sub(
        rf'({_LABEL_ALT})\s*:\s*:',
        r'\1 :',
        text,
        flags=re.IGNORECASE
    )


def fix_spaced_letters(text: str) -> str:
    text = re.sub(
        r'(?m)^([a-z] ){2,}[a-z]\s*$',
        lambda m: m.group(0).replace(' ', ''),
        text,
    )
    spaced_labels = [
        r'\s+'.join(list(lbl))
        for lbl in IDENTITY_LABELS
        if isinstance(lbl, str) and ' ' not in lbl and '/' not in lbl
    ]
    if spaced_labels:
        spaced_pattern = '|'.join(spaced_labels)
        text = re.sub(
            rf'(?<!\w)({spaced_pattern})(?=\s*:)',
            lambda m: m.group(0).replace(' ', ''),
            text,
            flags=re.IGNORECASE,
        )
    return text


_POST_IDENTITY_MARKERS = re.compile(
    r'(?<!\n)(terdakwa\s+(?:ditangkap|ditahan|tidak\s+ditahan|telah\s+ditahan|dalam\s+perkara|menghadap))',
    re.IGNORECASE,
)


def fix_leader_dots(text: str) -> str:
    text = re.sub(r'^(\s*\d{1,2})\.{2,}(\s*)', r'\1. ', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\.{3,}\s*:', r' :', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\.{3,}\s*', r' ', text, flags=re.MULTILINE)
    return text


def clean_text(raw_text: str, lowercase: bool = True) -> str:
    if not raw_text:
        return ""

    raw_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
    raw_text = normalize_unicode(raw_text)

    if lowercase:
        raw_text = raw_text.lower()

    raw_text = remove_watermark_lines(raw_text)
    raw_text = remove_headers_footers(raw_text)
    raw_text = fix_broken_label(raw_text)
    raw_text = fix_double_colon(raw_text)
    raw_text = fix_leader_dots(raw_text)
    raw_text = fix_spaced_letters(raw_text)
    raw_text = clean_identity_block(raw_text)
    raw_text = join_detention_list_markers(raw_text)
    raw_text = _POST_IDENTITY_MARKERS.sub(r'\n\1', raw_text)
    raw_text = re.sub(r'-{3,}', ' ', raw_text)
    raw_text = re.sub(r'\.{3,}', ' ', raw_text)

    lines = [ln.rstrip() for ln in raw_text.split('\n')]
    raw_text = '\n'.join(lines)
    raw_text = re.sub(r'[ \t]+', ' ', raw_text)
    raw_text = re.sub(r'\n{3,}', '\n\n', raw_text)
    raw_text = re.sub(r' +\n', '\n', raw_text)

    return raw_text.strip()

## PDF → Cleaning → Simpan ke /data/raw/

In [8]:
import datetime
import json

LOG_FILE = os.path.join(LOG_DIR, 'cleaning.log')

_QA_FIELD_PATTERNS = {
    'nama'        : r'\bnama(?:\s+lengkap)?\s*:',
    'tempat lahir': r'\btempat(?:/tgl\.?)?\s*lahir\s*:',
    'jenis kelamin': r'\bjenis\s+kelamin\s*:',
    'pekerjaan'   : r'\bpekerjaan\s*:',
}

_NOISE_PROBES = [
    'pelaksanaan fungsi peradilan',
    'kepaniteraan@mahkamahagung',
]


def log(message: str) -> None:
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f'[{timestamp}] {message}'
    print(entry)
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')


def qa_check_empty_values(text: str) -> list:
    empty = []

    merged_pattern = re.compile(
        rf'({_LABEL_ALT})\s*:\s*({_LABEL_ALT})\s*:',
        re.IGNORECASE
    )
    for m in merged_pattern.finditer(text):
        empty.append(m.group(1).strip())
    return empty


def qa_check(text: str) -> dict:
    found, missing = [], []
    for field, pattern in _QA_FIELD_PATTERNS.items():
        if re.search(pattern, text, re.IGNORECASE):
            found.append(field)
        else:
            missing.append(field)

    noise_found = [p for p in _NOISE_PROBES if p in text]
    empty_labels = qa_check_empty_values(text)

    return {
        'found': found,
        'missing': missing,
        'noise_leaked': noise_found,
        'empty_values': empty_labels,
        'passed': not missing and not noise_found,
    }


results = []

for i, pdf_file in enumerate(pdf_files, start=1):
    case_id  = f'case_{i:03d}'
    pdf_path = os.path.join(PDF_DIR, pdf_file)
    txt_path = os.path.join(RAW_DIR, f'{case_id}.txt')

    log(f'Memproses {case_id} <- {pdf_file}')

    raw_text = pdf_to_text(pdf_path)
    if not raw_text:
        log(f'  [SKIP] {case_id}: teks kosong setelah ekstraksi PDF')
        results.append({
            'case_id': case_id, 'file': pdf_file,
            'status': 'SKIP', 'word_count': 0, 'qa': None,
        })
        continue

    clean = clean_text(raw_text)

    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(clean)

    qa = qa_check(clean)
    wc = len(clean.split())

    if qa['passed']:
        log(f'  [OK] {case_id}: {wc} kata')
    else:
        issues = []
        if qa['missing']:
            issues.append(f'field tidak ditemukan: {qa["missing"]}')
        if qa['noise_leaked']:
            issues.append(f'noise tersisa: {qa["noise_leaked"]}')
        log(f'  [WARN] {case_id}: {wc} kata — {"; ".join(issues)}')

    results.append({
        'case_id': case_id, 'file': pdf_file,
        'status': 'OK', 'word_count': wc, 'qa': qa,
    })

ok        = sum(1 for r in results if r['status'] == 'OK')
skipped   = sum(1 for r in results if r['status'] == 'SKIP')
qa_failed = [r for r in results if r['status'] == 'OK' and r['qa'] and not r['qa']['passed']]

print(f'\n{"="*50}')
print(f'SELESAI: {ok}/{len(results)} berhasil, {skipped} dilewati')
if qa_failed:
    print(f'QA WARNING — {len(qa_failed)} file bermasalah:')
    for r in qa_failed:
        print(f'  {r["case_id"]} ({r["file"]})')
        if r['qa']['missing']:
            print(f'    field hilang   : {r["qa"]["missing"]}')
        if r['qa']['noise_leaked']:
            print(f'    noise tersisa  : {r["qa"]["noise_leaked"]}')
else:
    print('QA: semua file lulus.')
print(f'{"="*50}')

summary_path = os.path.join(LOG_DIR, 'processing_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f'Ringkasan disimpan: {summary_path}')

[2026-06-27 23:04:58] Memproses case_001 <- putusan_1014_pid.b_2019_pn_jkt.utr_20260627114755.pdf
[2026-06-27 23:04:58]   [OK] case_001: 4741 kata
[2026-06-27 23:04:58] Memproses case_002 <- putusan_1028_pid.b_2020_pn_jkt.utr_20260622233530.pdf
[2026-06-27 23:04:59]   [OK] case_002: 13170 kata
[2026-06-27 23:04:59] Memproses case_003 <- putusan_1050_pid.b_2021_pn_jkt.utr_20260612003949.pdf
[2026-06-27 23:04:59]   [OK] case_003: 11547 kata
[2026-06-27 23:04:59] Memproses case_004 <- putusan_1052_pid.b_2021_pn_jkt.utr_20260612004718.pdf
[2026-06-27 23:04:59]   [OK] case_004: 13012 kata
[2026-06-27 23:04:59] Memproses case_005 <- putusan_1099_pid.b_2020_pn_jkt.utr_20260627121743.pdf
[2026-06-27 23:05:00]   [OK] case_005: 24934 kata
[2026-06-27 23:05:00] Memproses case_006 <- putusan_1134_pid.b_2021_pn_jkt.utr_20260622223524.pdf
[2026-06-27 23:05:00]   [OK] case_006: 6169 kata
[2026-06-27 23:05:00] Memproses case_007 <- putusan_1204_pid.b_2021_pn_jkt.utr_20260622215259.pdf
[2026-06-27 23:0

## 4. Validasi Keutuhan Teks


In [9]:
import pandas as pd

df = pd.DataFrame(results)

median_wc = df[df['status'] == 'OK']['word_count'].median()
THRESHOLD = median_wc * 0.2

print(f'Median word count: {median_wc:.0f} kata')
print(f'Threshold minimum (20% median): {THRESHOLD:.0f} kata')
print()

df['valid'] = df.apply(
    lambda r: True if r['status'] == 'OK' and r['word_count'] >= THRESHOLD else False,
    axis=1
)

print('=== Hasil Validasi ===')
print(df[['case_id', 'file', 'status', 'word_count', 'valid']].to_string(index=False))
print()
print(f"Valid   : {df['valid'].sum()} dokumen")
print(f"Tidak valid: {(~df['valid']).sum()} dokumen")

Median word count: 11228 kata
Threshold minimum (20% median): 2246 kata

=== Hasil Validasi ===
 case_id                                                  file status  word_count  valid
case_001 putusan_1014_pid.b_2019_pn_jkt.utr_20260627114755.pdf     OK        4741   True
case_002 putusan_1028_pid.b_2020_pn_jkt.utr_20260622233530.pdf     OK       13170   True
case_003 putusan_1050_pid.b_2021_pn_jkt.utr_20260612003949.pdf     OK       11547   True
case_004 putusan_1052_pid.b_2021_pn_jkt.utr_20260612004718.pdf     OK       13012   True
case_005 putusan_1099_pid.b_2020_pn_jkt.utr_20260627121743.pdf     OK       24934   True
case_006 putusan_1134_pid.b_2021_pn_jkt.utr_20260622223524.pdf     OK        6169   True
case_007 putusan_1204_pid.b_2021_pn_jkt.utr_20260622215259.pdf     OK        4342   True
case_008  putusan_124_pid.b_2020_pn_jkt.utr_20260627120924.pdf     OK       15199   True
case_009 putusan_1298_pid.b_2019_pn_jkt.utr_20260622234946.pdf     OK        7727   True
case_010 putus

In [10]:
invalid = df[~df['valid']]
if len(invalid) > 0:
    print('Dokumen yang tidak lolos validasi:')
    print(invalid[['case_id', 'file', 'word_count']].to_string(index=False))
    print('\n=> Cek manual PDF-nya: kemungkinan scan/gambar, bukan teks.')
else:
    print('Semua dokumen lolos validasi.')

Semua dokumen lolos validasi.


## 5. Simpan Index Dokumen

In [11]:
INDEX_PATH = os.path.join(BASE_DIR, 'data', 'index.csv')
df.to_csv(INDEX_PATH, index=False, encoding='utf-8')
print(f'Index disimpan: {INDEX_PATH}')
print(f'Total dokumen valid siap diproses tahap 2: {df["valid"].sum()}')

Index disimpan: C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\data\index.csv
Total dokumen valid siap diproses tahap 2: 36


## 6. Cek Sampel Output


In [12]:
valid_cases = df[df['valid']]['case_id'].tolist()

if not valid_cases:
    print('Tidak ada dokumen valid.')
else:
    for case_id in valid_cases[:5]:
        sample_path = os.path.join(RAW_DIR, f'{case_id}.txt')
        try:
            with open(sample_path, 'r', encoding='utf-8') as f:
                sample = f.read()
            print(f'\n{"="*60}')
            print(f'=== Sampel: {case_id} ===')
            print(f'{"="*60}')
            print(sample[:2000])
            print('...')
        except FileNotFoundError:
            print(f'[MISSING] {case_id}: file tidak ditemukan')


=== Sampel: case_001 ===
putusan
nomor 1014/pid.b/2019/pn jkt.utr
demi keadilan berdasarkan ketuhanan yang maha esa;
pengadilan negeri jakarta utara yang mengadili perkara pidana dengan
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
berikut dalam perkara para terdakwa:
terdakwa i
nama lengkap : eka kurniawan
tempat lahir : jakarta
umur/tanggal lahir : 40 tahun / 15 januari 1979
jenis kelamin : laki-laki
kewarganegaraan : indonesia
tempat tinggal : jalan kampung beting rt.04/rw.09 kel. semper barat kec. cilincing jakarta utara
agama : islam
pekerjaan : pns (dishub)
pendidikan : smk
terdakwa ii
nama lengkap : restiana binti rosano boyke
tempat lahir : jakarta
umur/tanggal lahir : 30 tahun / 09 oktober 1988
jenis kelamin : laki-laki
kewarganegaraan : indonesia
tempat tinggal : jalan manggar blok y gg.i/26 rt.04/rw.08 kel. lagoa kec. koja jakarta utara
agama : islam
pekerjaan : ibu rumah tangga
pendidikan : smk
terdakwa dalam perkara ini telah ditahan berdasarka

## 7. Clean Amar

In [13]:
_AMAR_START = re.compile(
    r'(?:^|\n)'
    r'[ \t]*'
    r'm[\s.]*e[\s.]*n[\s.]*g[\s.]*a[\s.]*d[\s.]*i[\s.]*l[\s.]*i'
    r'(?!\s+perkara)'
    r'[ \t]*:?[ \t]*(?:\n|$)',
    re.IGNORECASE,
)

_AMAR_END = re.compile(
    r'\n[ \t]*(?:'
    r'demikian(?:lah)?\s+(?:diputus|putusan)'
    r'|ditetapkan\s+di'
    r'|dijatuhkan\s+(?:di|pada|dalam)'
    r'|putusan\s+ini\s+diucapkan'
    r'|hakim\s+(?:ketua|anggota)'
    r'|(?:jakarta|bandung|surabaya|medan|depok|bekasi|bogor|tangerang),\s*\d'
    r'|demikianlah\s+diputuskan'
    r'|diucapkan\s+dalam\s+sidang'
    r')',
    re.IGNORECASE,
)

_PASCA_AMAR = re.compile(
    r'(?:demikian(?:lah)?\s+diputuskan'
    r'|diucapkan\s+dalam\s+(?:sidang|persidangan)'
    r'|hakim(?:-hakim)?\s+anggota'
    r'|panitera\s+pengganti'
    r')',
    re.IGNORECASE,
)

_SELA_SIGNALS = re.compile(
    r'(?:'
    r'eksepsi\s+(?:atau\s+keberatan\s+)?(?:dari\s+)?penasihat\s+hukum\s+terdakwa\s+.{0,80}tidak\s+dapat\s+diterima'
    r'|memerintahkan\s+penuntut\s+umum\s+untuk\s+melanjutkan\s+pemeriksaan\s+perkara'
    r'|menangguhkan\s+biaya\s+perkara\s+sampai\s+dengan\s+putusan\s+akhir'
    r')',
    re.IGNORECASE,
)

_FINAL_SIGNALS = re.compile(
    r'(?:'
    r'menjatuhkan\s+pidana'
    r'|membebaskan\s+terdakwa'
    r'|melepaskan\s+terdakwa'
    r'|membebankan\s+(?:kepada\s+)?(?:para\s+)?terdakwa\s+(?:untuk\s+)?membayar\s+biaya\s+perkara'
    r'|menolak\s+(?:seluruh\s+)?(?:gugatan|dakwaan|tuntutan)'
    r')',
    re.IGNORECASE,
)

_INLINE_BUTIR = re.compile(
    r'(?<!\d)'
    r'(?<!\w)'
    r'(\s*\d{1,2}[.)]'
    r'\s+(?=\S))',
    re.IGNORECASE,
)

_KATA_BUTIR = (
    r'menyatakan|menjatuhkan|menetapkan|memerintahkan|'
    r'membebankan|memulihkan|membebaskan|melepaskan|menolak|mengabulkan|menghukum'
)

_BUTIR_FULL = re.compile(r'^\s*\d{1,2}[.)]\s*\S')
_BUTIR_ONLY = re.compile(r'^\s*\d{1,2}[.)]\s*$')
_SUB_BUTIR  = re.compile(r'^\s*(?:\d{1,2}|[a-z])[.)]\s*$')


def extract_amar(text: str, case_id: str = '', debug: bool = False) -> str:
    prefix = f'[{case_id}] ' if case_id else ''

    all_starts = list(_AMAR_START.finditer(text))
    if not all_starts:
        if debug:
            print(f'{prefix}[DEBUG] Header "mengadili" TIDAK DITEMUKAN.')
        return ''

    if debug:
        print(f'{prefix}[DEBUG] Ditemukan {len(all_starts)} blok "mengadili"')

    chosen_start = None
    for m_start in reversed(all_starts):
        amar_start = m_start.end()
        tail = text[amar_start:]

        m_end = _AMAR_END.search(tail)
        candidate = tail[:m_end.start()].strip() if m_end else tail.strip()

        head = candidate[:800]

        is_sela  = bool(_SELA_SIGNALS.search(head))
        is_final = bool(_FINAL_SIGNALS.search(candidate))

        if debug:
            print(
                f'{prefix}[DEBUG] Blok di char {m_start.start()}: '
                f'is_sela={is_sela}, is_final={is_final}'
            )

        if is_final and not is_sela:
            chosen_start = m_start
            if debug:
                print(f'{prefix}[DEBUG] => Dipilih sebagai putusan akhir')
            break

        if is_final and chosen_start is None:
            chosen_start = m_start
            if debug:
                print(f'{prefix}[DEBUG] => Dipilih sebagai fallback (ada sela tapi ada final juga)')

    if chosen_start is None:
        chosen_start = all_starts[-1]
        if debug:
            print(f'{prefix}[DEBUG] => Tidak ada yang cocok, pakai blok terakhir sebagai fallback')

    amar_start = chosen_start.end()
    if debug:
        print(f'{prefix}[DEBUG] Amar START di char {chosen_start.start()}: '
              f'{repr(text[chosen_start.start():chosen_start.end()])}')

    tail  = text[amar_start:]
    m_end = _AMAR_END.search(tail)

    if m_end:
        amar_end = amar_start + m_end.start()
        if debug:
            print(f'{prefix}[DEBUG] Amar END di char {amar_end}: {repr(tail[m_end.start():m_end.start()+80])}')
    else:
        amar_end = len(text)
        if debug:
            print(f'{prefix}[DEBUG] Amar END tidak ditemukan — ambil sampai akhir.')

    amar = text[amar_start:amar_end].strip()
    if debug:
        print(f'{prefix}[DEBUG] Panjang amar: {len(amar)} karakter, {len(amar.split())} kata\n')

    return amar


def split_inline_butir(text: str) -> str:
    return re.sub(
        rf'\s+(?=\d{{1,2}}\.(?:{_KATA_BUTIR}))',
        '\n',
        text,
        flags=re.IGNORECASE,
    )


def normalize_amar(amar: str) -> str:
    amar = split_inline_butir(amar)

    m = _PASCA_AMAR.search(amar)
    if m:
        preceding = amar[:m.start()]
        if _FINAL_SIGNALS.search(preceding):
            amar = preceding.rstrip()

    amar = split_inline_butir(amar)

    lines = amar.splitlines()
    out   = []
    i     = 0

    while i < len(lines):
        line = lines[i].strip()

        if not line:
            i += 1
            continue

        if _BUTIR_ONLY.match(line) or _SUB_BUTIR.match(line):
            j = i + 1
            while j < len(lines) and not lines[j].strip():
                j += 1
            if j < len(lines):
                out.append(line + ' ' + lines[j].strip())
                i = j + 1
            else:
                out.append(line)
                i += 1
            continue

        if _BUTIR_FULL.match(line):
            out.append(line)
            i += 1
            continue

        if out:
            out[-1] = out[-1] + ' ' + line
        else:
            out.append(line)
        i += 1

    return '\n'.join(out).strip()


def debug_amar_all(raw_dir: str, max_files: int = None, show_output: bool = True):
    txt_files = sorted(f for f in os.listdir(raw_dir) if f.endswith('.txt'))
    if max_files:
        txt_files = txt_files[:max_files]

    print(f'Total file: {len(txt_files)}\n{"="*60}')

    results, not_found = [], []

    for fname in txt_files:
        case_id = fname.replace('.txt', '')
        with open(os.path.join(raw_dir, fname), 'r', encoding='utf-8') as f:
            text = f.read()

        print(f'\n{"─"*60}\nCASE: {case_id}')
        amar = normalize_amar(extract_amar(text, case_id=case_id, debug=True))

        if not amar:
            not_found.append(case_id)
        else:
            results.append({'case_id': case_id, 'amar': amar})
            if show_output:
                print('--- AMAR OUTPUT ---')
                print(amar[:5000])

    print(f'\n{"="*60}')
    print(f'Amar ditemukan : {len(results)} file')
    print(f'Amar TIDAK ada : {len(not_found)} file')
    if not_found:
        print(f'Perlu cek manual: {not_found}')

    return results

amar_results = debug_amar_all(RAW_DIR, max_files=36)

Total file: 36

────────────────────────────────────────────────────────────
CASE: case_001
[case_001] [DEBUG] Ditemukan 1 blok "mengadili"
[case_001] [DEBUG] Blok di char 31239: is_sela=False, is_final=True
[case_001] [DEBUG] => Dipilih sebagai putusan akhir
[case_001] [DEBUG] Amar START di char 31239: '\nmengadili\n'
[case_001] [DEBUG] Amar END di char 32357: '\ndemikian diputuskan dalam musyawarah majelis hakim pengadilan negeri\njakarta ut'
[case_001] [DEBUG] Panjang amar: 1107 karakter, 156 kata

--- AMAR OUTPUT ---
1. menyatakan terdakwa 1.eka kurniawan dan terdakwa 2.restiani binti rosano boyke, telah terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana “turut serta melakukan menyuruh menempatkan keterangan palsu ke dalam surat autentik”;
2. menjatuhkan pidana kepada terdakwa 1.eka kurniawan, oleh karena itu dengan pidana penjara selama 1 (satu) tahun dan terdakwa 2.restiani binti rosano boyke, oleh karena itu dengan pidana penjara selama 9 (sembilan) bulan;
3. me

## Save Amar

In [14]:
AMAR_DIR = os.path.join(BASE_DIR, 'data', 'amar')
os.makedirs(AMAR_DIR, exist_ok=True)

amar_saved = []
missing_amar = []

for fname in sorted(f for f in os.listdir(RAW_DIR) if f.endswith('.txt')):
    case_id = fname.replace('.txt', '')
    with open(os.path.join(RAW_DIR, fname), 'r', encoding='utf-8') as f:
        text = f.read()

    amar = normalize_amar(extract_amar(text, case_id=case_id, debug=False))

    if amar:
        amar_path = os.path.join(AMAR_DIR, fname)
        with open(amar_path, 'w', encoding='utf-8') as f:
            f.write(amar)
        amar_saved.append({'case_id': case_id, 'amar': amar})
    else:
        missing_amar.append(case_id)
        print(f'[SKIP] {case_id}: amar tidak ditemukan')

print(f'\nAmar disimpan  : {len(amar_saved)} file → {AMAR_DIR}')
print(f'Amar tidak ada : {len(missing_amar)} file')
if missing_amar:
    print(f'Perlu cek manual: {missing_amar}')


Amar disimpan  : 36 file → C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\data\amar
Amar tidak ada : 0 file
